In [ ]:
# ── CELL 1 — Constants + install dependencies ──────────────────────────────
#
# Branch: ctclip-stage2 — DO NOT MERGE TO MAIN.
#
# Extracts CT-CLIP visual features for the full CT-RATE dataset and uploads
# them to Google Drive via rclone. Fully resumable: a crash, disconnect or
# OOM loses at most FLUSH_EVERY volumes of work.
#
# Model init and preprocessing are copied verbatim from
# ctclip_feasibility_test.py in this repo - do not rewrite them here.
#
# SCALE WARNING: the full dataset is ~50,188 volumes (47,149 train + 3,039
# valid). Extracting all of it means DOWNLOADING ~7.8 TB of raw CT from
# HuggingFace to produce ~677 GB of features. The download - not the GPU - is
# the bottleneck, and this will span many sessions. That's why every part of
# this notebook is resumable.

import os
from pathlib import Path

# ---- User-tunable constants ----
RCLONE_REMOTE = "gdrive2:"      # UPDATE to match your configured rclone remote
HF_TOKEN      = os.environ.get("HF_TOKEN")
STAGING_DIR   = "/content/staging"
BATCH_SIZE    = 4               # reduce to 2 or 1 if VRAM OOM
FLUSH_EVERY   = 500             # rclone upload cadence (volumes, not batches)
LOG_EVERY     = 20              # progress-line cadence (volumes)
TARGET_SHAPE  = (480, 480, 240)

# RCLONE_REMOTE already ends in ':', so remote paths concatenate straight onto
# it - do NOT add a second colon.
REMOTE_ROOT       = RCLONE_REMOTE + "ctrate_features"
REMOTE_CHECKPOINT = REMOTE_ROOT + "/checkpoint.json"
LOCAL_CHECKPOINT  = Path("/content/checkpoint.json")
LOG_FILE          = Path("/content/extract_log.txt")

HF_REPO = "ibrahimhamamci/CT-RATE"
# Real CT-RATE v2 layout, verified empirically in this project:
#   dataset/train_fixed/{patient}/{scan}/{volume}.nii.gz
#   e.g. dataset/train_fixed/train_1/train_1_a/train_1_a_1.nii.gz
SPLITS = {"train": "dataset/train_fixed", "valid": "dataset/valid_fixed"}

CT_CLIP_REPO_URL  = "https://github.com/ibrahimethemhamamci/CT-CLIP.git"
CT_CLIP_LOCAL_DIR = Path("./CT-CLIP")
WORK_DIR     = Path("./ctclip_work")
WEIGHTS_PATH = WORK_DIR / "CT-CLIP_v2.pt"

# CTViT / CTCLIP construction config, from CT-CLIP's scripts/run_zero_shot.py.
CTVIT_CONFIG = dict(
    dim=512,
    codebook_size=8192,
    image_size=480,
    patch_size=20,
    temporal_patch_size=10,
    spatial_depth=4,
    temporal_depth=4,
    dim_head=32,
    heads=8,
)
CTCLIP_CONFIG = dict(
    dim_image=294912,
    dim_text=768,
    dim_latent=512,
    extra_latent_projection=False,
    use_mlm=False,
    downsample_image_embeds=False,
    use_all_token_embeds=False,
)
TEXT_ENCODER_NAME = "microsoft/BiomedVLP-CXR-BERT-specialized"

assert HF_TOKEN, "Set HF_TOKEN (a token that has accepted the CT-RATE gated terms)."

WORK_DIR.mkdir(parents=True, exist_ok=True)
Path(STAGING_DIR).mkdir(parents=True, exist_ok=True)

# ---- Install dependencies ----
import subprocess
import sys


def _pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


_pip_install("nibabel", "huggingface_hub", "transformers", "torch")

if not CT_CLIP_LOCAL_DIR.exists():
    subprocess.run(["git", "clone", CT_CLIP_REPO_URL, str(CT_CLIP_LOCAL_DIR)], check=True)
else:
    print(f"{CT_CLIP_LOCAL_DIR} already exists, skipping clone.")

# CT-CLIP has NO setup.py at its repo root: transformer_maskgit (CTViT) and
# CT_CLIP (the CTCLIP wrapper) are each their own installable subdirectory,
# each nesting the real package one level deeper. Do NOT add the repo root to
# sys.path - the outer transformer_maskgit/ folder has no __init__.py and would
# shadow the real install as an empty namespace package.
for sub in ("transformer_maskgit", "CT_CLIP"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(CT_CLIP_LOCAL_DIR / sub)],
        check=True,
    )

print("Dependencies installed.")
print("If this was a fresh install, RESTART THE RUNTIME now, then re-run CELL 1")
print("before continuing - a failed import caches a broken module in sys.modules.")


In [ ]:
# ── CELL 2 — rclone config ─────────────────────────────────────────────────

import pathlib

# ===========================================================================
# TODO: PASTE YOUR rclone.conf CONTENT INTO THE STRING BELOW.
#
# Get it by running `rclone config show` on a machine where the remote already
# works. It should look like:
#
#   [gdrive2]
#   type = drive
#   scope = drive
#   token = {"access_token":"...","refresh_token":"...","expiry":"..."}
#
# The section name in brackets must match RCLONE_REMOTE from CELL 1 (minus the
# trailing colon) - i.e. "gdrive2" for RCLONE_REMOTE = "gdrive2:".
#
# SECURITY: this string contains a live OAuth token. Do NOT commit this
# notebook with it filled in, and do not share the executed notebook.
# ===========================================================================
RCLONE_CONF = """
"""  # <-- PASTE BETWEEN THE TRIPLE QUOTES

assert RCLONE_CONF.strip(), "Paste your rclone.conf content into RCLONE_CONF above."

cfg = pathlib.Path.home() / ".config/rclone/rclone.conf"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text(RCLONE_CONF)
print(f"Wrote {cfg}")

# Verify the remote actually resolves before spending hours extracting.
!rclone lsd {RCLONE_REMOTE} | head


In [ ]:
# ── CELL 3 — Build and load CT-CLIP ────────────────────────────────────────
# Copied verbatim from ctclip_feasibility_test.py.

# ---- Download CT-CLIP_v2.pt weights ----
if WEIGHTS_PATH.exists():
    print(f"Weights already downloaded at {WEIGHTS_PATH}, skipping.")
else:
    from huggingface_hub import HfApi, hf_hub_download

    api = HfApi()
    all_files = api.list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
    candidates = [f for f in all_files if "CT-CLIP_v2" in f and f.endswith(".pt")]
    if not candidates:
        raise FileNotFoundError(
            f"No file matching 'CT-CLIP_v2*.pt' found in {HF_REPO}."
        )
    remote_path = candidates[0]
    print(f"Found weights at: {remote_path}")

    local_path = Path(
        hf_hub_download(
            repo_id=HF_REPO,
            repo_type="dataset",
            filename=remote_path,
            token=HF_TOKEN,
            local_dir=str(WORK_DIR),
        )
    )
    if local_path != WEIGHTS_PATH:
        local_path.rename(WEIGHTS_PATH)
print(f"Weights ready at {WEIGHTS_PATH}")

# ---- Build model ----
import torch
from transformer_maskgit import CTViT
from transformers import BertModel, BertTokenizer

try:
    from ct_clip import CTCLIP
except ImportError as exc:
    raise ImportError(
        "Could not import CTCLIP from the ct_clip package - check "
        "CT-CLIP/scripts/run_zero_shot.py in the cloned repo for the current "
        "correct import path if the package layout has changed."
    ) from exc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

text_tokenizer = BertTokenizer.from_pretrained(TEXT_ENCODER_NAME)
text_encoder = BertModel.from_pretrained(TEXT_ENCODER_NAME)
# Text encoder is required for CTCLIP's __init__ but not used at inference -
# only clip.visual_transformer(...) is called below.

image_encoder = CTViT(**CTVIT_CONFIG)

clip = CTCLIP(
    image_encoder=image_encoder,
    text_encoder=text_encoder,
    tokenizer=text_tokenizer,
    **CTCLIP_CONFIG,
)

# Don't use clip.load(): it does a strict load_state_dict, which rejects the
# whole checkpoint over version drift. transformers >=4.31 removed the
# `position_ids` buffer from BertModel's embeddings, but CT-CLIP_v2.pt was
# saved when it still existed. It's a non-learned arange() buffer, so dropping
# it loses nothing.
state_dict = torch.load(str(WEIGHTS_PATH), map_location="cpu")
state_dict.pop("text_transformer.embeddings.position_ids", None)

# Load non-strictly, but REPORT what didn't match. A silently-missing
# visual_transformer.* key would leave that part of the encoder randomly
# initialised and make every extracted feature meaningless.
missing, unexpected = clip.load_state_dict(state_dict, strict=False)
if missing:
    print(f"WARNING - {len(missing)} missing key(s), first few: {missing[:5]}")
if unexpected:
    print(f"WARNING - {len(unexpected)} unexpected key(s), first few: {unexpected[:5]}")
if not missing and not unexpected:
    print("All checkpoint keys matched exactly.")

visual_missing = [k for k in missing if k.startswith("visual_transformer")]
if visual_missing:
    raise RuntimeError(
        f"{len(visual_missing)} visual_transformer key(s) missing from the "
        f"checkpoint (e.g. {visual_missing[:3]}). The visual encoder would be "
        "partly random and every extracted feature meaningless - fix this "
        "before trusting any output from this notebook."
    )

clip = clip.to(DEVICE)
clip.eval()
for p in clip.parameters():
    p.requires_grad_(False)
print("CTCLIP model built and weights loaded.")

# ---- Preprocessing (copied verbatim from ctclip_feasibility_test.py) ----
import nibabel as nib
import numpy as np


def center_crop_or_pad(arr: np.ndarray, target_shape: tuple[int, int, int]) -> np.ndarray:
    """Crop (centered) or zero-pad (centered) each axis independently to
    reach target_shape. Operates on an (H, W, D)-ordered array."""
    out = arr
    for axis, target in enumerate(target_shape):
        cur = out.shape[axis]
        if cur > target:
            start = (cur - target) // 2
            idx = [slice(None)] * out.ndim
            idx[axis] = slice(start, start + target)
            out = out[tuple(idx)]
        elif cur < target:
            pad_total = target - cur
            pad_before = pad_total // 2
            pad_after = pad_total - pad_before
            pad_width = [(0, 0)] * out.ndim
            pad_width[axis] = (pad_before, pad_after)
            out = np.pad(out, pad_width, mode="constant", constant_values=0.0)
    return out


def load_and_preprocess(path: Path) -> torch.Tensor:
    img = nib.load(str(path))
    arr = np.asarray(img.dataobj, dtype=np.float32)  # raw NIfTI axis order: (H, W, D)

    arr = np.clip(arr, -1000.0, 1000.0) / 1000.0  # HU clip -> [-1, 1]
    arr = center_crop_or_pad(arr, TARGET_SHAPE)  # (480, 480, 240)

    arr = arr.transpose(2, 0, 1)  # (H, W, D) -> (D, H, W)
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)  # (1, 1, 240, 480, 480)
    return tensor.float()


In [ ]:
# ── CELL 4 — Resume checkpoint + logging ───────────────────────────────────

import json
import subprocess
import time


def log(msg: str) -> None:
    """Print AND append to a persistent log file. The log is uploaded next to
    the checkpoint on every flush, so history survives a session dying."""
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG_FILE, "a") as fh:
        fh.write(line + "\n")


# --ignore-errors so a missing checkpoint (first ever run) isn't fatal.
subprocess.run(
    ["rclone", "copy", REMOTE_CHECKPOINT, "/content/", "--ignore-errors"],
    check=False,
)
# Pull any previous log too, so this session appends to the running history
# rather than starting a fresh one.
subprocess.run(
    ["rclone", "copy", REMOTE_ROOT + "/extract_log.txt", "/content/", "--ignore-errors"],
    check=False,
)

if LOCAL_CHECKPOINT.exists():
    ckpt = json.loads(LOCAL_CHECKPOINT.read_text())
    done = set(ckpt.get("done", []))
    log(f"RESUME: checkpoint found - {len(done):,} volume(s) already extracted.")
else:
    done = set()
    log("RESUME: no checkpoint on the remote - starting fresh.")


def save_checkpoint(done_set: set) -> None:
    """Write the checkpoint + log locally and mirror both to the remote."""
    LOCAL_CHECKPOINT.write_text(json.dumps({"done": sorted(done_set)}))
    subprocess.run(["rclone", "copy", str(LOCAL_CHECKPOINT), REMOTE_ROOT + "/"], check=False)
    if LOG_FILE.exists():
        subprocess.run(["rclone", "copy", str(LOG_FILE), REMOTE_ROOT + "/"], check=False)


In [ ]:
# ── CELL 5 — Volume discovery ──────────────────────────────────────────────

from huggingface_hub import HfApi

api = HfApi()

# One recursive tree listing per split, rather than walking ~20k directories
# with individual ls calls. Metadata-only (no downloads) - the same approach
# scripts/check_dataset_size.py uses in this repo. `size` comes back with each
# entry, so we get the real raw-byte total for free.
all_volumes = []  # list of (split, hf_repo_path, volname)
raw_bytes_all = 0
for split, root in SPLITS.items():
    print(f"Listing {root} ...")
    n_split = 0
    for entry in api.list_repo_tree(
        HF_REPO, repo_type="dataset", path_in_repo=root,
        recursive=True, token=HF_TOKEN,
    ):
        path = getattr(entry, "path", "")
        if path.endswith(".nii.gz"):
            volname = Path(path).name[: -len(".nii.gz")]
            all_volumes.append((split, path, volname))
            raw_bytes_all += getattr(entry, "size", 0) or 0
            n_split += 1
    log(f"DISCOVERY: {split:5s} -> {n_split:,} volume(s)")

remaining = [v for v in all_volumes if v[2] not in done]
raw_gb_all = raw_bytes_all / 1e9
pct_done = (100 * len(done) / len(all_volumes)) if all_volumes else 0.0

log(f"DISCOVERY: total={len(all_volumes):,} done={len(done):,} ({pct_done:.1f}%) "
    f"remaining={len(remaining):,}")
log(f"DISCOVERY: raw dataset size = {raw_gb_all:.1f} GB "
    f"({raw_gb_all / max(len(all_volumes), 1) * 1000:.0f} MB/volume avg) - this is "
    f"how much must be downloaded from HF in total across all sessions.")


In [ ]:
# ── CELL 6 — Batched extraction loop ───────────────────────────────────────

import shutil

from huggingface_hub import HfFileSystem

fs = HfFileSystem(token=HF_TOKEN)

# Session counters
uploaded_bytes = 0          # bytes actually pushed to the remote this session
feature_bytes_total = 0     # bytes of .pt features produced this session
raw_bytes_downloaded = 0    # bytes of raw NIfTI pulled from HF this session
processed_this_session = 0
since_flush = 0
since_log = 0
dl_seconds = 0.0
pre_seconds = 0.0
enc_seconds = 0.0
oom_fallbacks = 0
t_start = time.time()


def flush_staging() -> tuple:
    """Upload everything staged so far, clear staging, persist checkpoint+log.
    Returns (n_files, gb_uploaded)."""
    global uploaded_bytes
    staged = list(Path(STAGING_DIR).rglob("*.pt"))
    n, gb = len(staged), 0.0
    if staged:
        batch_bytes = sum(f.stat().st_size for f in staged)
        uploaded_bytes += batch_bytes
        gb = batch_bytes / 1e9
        subprocess.run(
            ["rclone", "copy", STAGING_DIR, REMOTE_ROOT + "/", "--progress"],
            check=True,
        )
        for split_dir in Path(STAGING_DIR).iterdir():
            if split_dir.is_dir():
                shutil.rmtree(split_dir)
    save_checkpoint(done)
    return n, gb


def encode(tensor):
    """Encode (B, 1, 240, 480, 480) -> (B, 24, 24, 24, 512)."""
    with torch.no_grad():
        return clip.visual_transformer(tensor.to(DEVICE), return_encoded_tokens=True)


batches = [remaining[i : i + BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]
log(f"START: {len(remaining):,} volume(s) to go, in {len(batches):,} batch(es) "
    f"of up to {BATCH_SIZE}. Flushing every {FLUSH_EVERY} volumes.")

for batch in batches:
    tmp_paths = []

    # (a) download each NIfTI in the batch
    t0 = time.perf_counter()
    for i, (split, hf_path, volname) in enumerate(batch):
        tmp = Path(f"/content/tmp_{i}.nii.gz")
        with fs.open(f"datasets/{HF_REPO}/{hf_path}", "rb") as src, open(tmp, "wb") as dst:
            dst.write(src.read())
        raw_bytes_downloaded += tmp.stat().st_size
        tmp_paths.append(tmp)
    dl_seconds += time.perf_counter() - t0

    # (b) preprocess each -> list of (1, 1, 240, 480, 480)
    t0 = time.perf_counter()
    tensors = [load_and_preprocess(p) for p in tmp_paths]
    # (c) each tensor already carries a leading batch dim, so cat (not stack)
    # to get (B, 1, 240, 480, 480) - stack would add a spurious 6th dim.
    batch_tensor = torch.cat(tensors, dim=0)
    pre_seconds += time.perf_counter() - t0

    # (d)/(e) encode, falling back to one-by-one on OOM
    t0 = time.perf_counter()
    try:
        features = encode(batch_tensor).cpu()
    except torch.cuda.OutOfMemoryError:
        oom_fallbacks += 1
        log(f"WARNING: CUDA OOM at BATCH_SIZE={len(batch)} - retrying this batch "
            f"one volume at a time. Lower the BATCH_SIZE constant in CELL 1 to "
            f"stop paying this retry cost on every batch. "
            f"(OOM fallbacks so far: {oom_fallbacks})")
        del batch_tensor
        torch.cuda.empty_cache()
        feats = []
        for t in tensors:
            feats.append(encode(t).cpu())
            torch.cuda.empty_cache()
        features = torch.cat(feats, dim=0)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    enc_seconds += time.perf_counter() - t0

    # (f) save each feature under its ORIGINAL volume name, so training-time
    # code can find it from the reports CSV VolumeName with a plain
    # volname + ".pt" lookup. Never index numbers.
    for i, (split, hf_path, volname) in enumerate(batch):
        out_path = Path(STAGING_DIR) / split / f"{volname}.pt"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(features[i].half(), out_path)
        feature_bytes_total += out_path.stat().st_size

    # (g) drop the temp NIfTIs immediately - they're ~119-156 MB each
    for tmp in tmp_paths:
        tmp.unlink(missing_ok=True)
    del features, tensors
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # (h) mark done
    for split, hf_path, volname in batch:
        done.add(volname)
    processed_this_session += len(batch)
    since_flush += len(batch)
    since_log += len(batch)

    # progress line
    if since_log >= LOG_EVERY:
        since_log = 0
        elapsed = time.time() - t_start
        rate = processed_this_session / elapsed if elapsed > 0 else 0.0
        left = len(all_volumes) - len(done)
        eta_h = (left / rate / 3600) if rate > 0 else float("nan")
        free_gb = shutil.disk_usage("/content").free / 1e9
        log(
            f"[{len(done):,}/{len(all_volumes):,} {100 * len(done) / len(all_volumes):4.1f}%] "
            f"feat={feature_bytes_total / 1e9:.2f}GB up={uploaded_bytes / 1e9:.2f}GB "
            f"raw_dl={raw_bytes_downloaded / 1e9:.1f}GB | "
            f"dl={dl_seconds / processed_this_session:.1f}s "
            f"pre={pre_seconds / processed_this_session:.1f}s "
            f"enc={enc_seconds / processed_this_session:.1f}s /vol | "
            f"{rate * 3600:.0f}vol/h | elapsed={elapsed / 3600:.2f}h ETA={eta_h:.1f}h | "
            f"disk={free_gb:.0f}GB free"
        )

    # (i) flush on a volume (not batch) cadence
    if since_flush >= FLUSH_EVERY:
        since_flush = 0
        n_files, gb = flush_staging()
        log(f"FLUSH: uploaded {gb:.2f}GB ({n_files} file(s)) | "
            f"session total {uploaded_bytes / 1e9:.2f}GB | "
            f"checkpoint saved at {len(done):,} done")


In [ ]:
# ── CELL 7 — Final flush + summary ─────────────────────────────────────────

n_files, gb = flush_staging()
log(f"FINAL FLUSH: uploaded {gb:.2f}GB ({n_files} file(s))")

elapsed = time.time() - t_start
log("=" * 70)
log("SESSION COMPLETE")
log(f"  volumes processed this session: {processed_this_session:,}")
log(f"  cumulative done:                {len(done):,} / {len(all_volumes):,} "
    f"({100 * len(done) / len(all_volumes):.1f}%)")
log(f"  remaining:                      {len(all_volumes) - len(done):,}")
log(f"  features produced this session: {feature_bytes_total / 1e9:.2f} GB")
log(f"  uploaded this session:          {uploaded_bytes / 1e9:.2f} GB")
log(f"  raw downloaded this session:    {raw_bytes_downloaded / 1e9:.1f} GB")
log(f"  wall time:                      {elapsed / 3600:.2f} h")
if processed_this_session:
    log(f"  average:                        {elapsed / processed_this_session:.2f} s/volume")
    log(f"    of which download:            {dl_seconds / processed_this_session:.2f} s")
    log(f"    of which preprocess:          {pre_seconds / processed_this_session:.2f} s")
    log(f"    of which encode:              {enc_seconds / processed_this_session:.2f} s")
log(f"  OOM fallbacks:                  {oom_fallbacks}")
log("=" * 70)
log(f"Features: {REMOTE_ROOT}/train/ and {REMOTE_ROOT}/valid/")
log(f"Checkpoint: {REMOTE_CHECKPOINT}")
log(f"Log: {REMOTE_ROOT}/extract_log.txt")
